# 02. L1 표현 예측과 EMA 교사 실습

목표: V-JEPA가 사용하는 L1 표현 손실의 성질과, 경사로 직접 학습하지 않는 EMA 타깃 인코더의 움직임을 작은 스칼라 예제로 확인합니다.

In [ ]:
targets = [0.9, 1.0, 1.1, 1.0, 8.0]  # 마지막 값은 이상치입니다.
mean_target = sum(targets) / len(targets)
median_target = sorted(targets)[len(targets) // 2]

def l1(candidate):
    return sum(abs(candidate - value) for value in targets) / len(targets)

def l2(candidate):
    return sum((candidate - value) ** 2 for value in targets) / len(targets)

print({'평균(L2 최적점)': mean_target, '중앙값(L1 최적점)': median_target})
print({'평균점의 L1': l1(mean_target), '중앙값의 L1': l1(median_target)})
print({'평균점의 L2': l2(mean_target), '중앙값의 L2': l2(median_target)})

In [ ]:
# 온라인 인코더와 predictor는 손실로 갱신하고, target은 EMA로만 갱신합니다.
online, predictor, target = 0.15, 0.10, 1.0
learning_rate, momentum = 0.04, 0.95
contexts = [0.4, 0.7, 1.0, 1.3, 1.6] * 40
history = []

for context in contexts:
    target_feature = target * (1.5 * context + 0.2)
    prediction = predictor * online * context
    error = prediction - target_feature
    sign = 1.0 if error > 0 else -1.0 if error < 0 else 0.0
    old_online = online
    # L1의 부분경사로 두 학습 파라미터를 갱신합니다.
    online -= learning_rate * sign * predictor * context
    predictor -= learning_rate * sign * old_online * context
    target = momentum * target + (1.0 - momentum) * online
    history.append(abs(error))

first = sum(history[:20]) / 20
last = sum(history[-20:]) / 20
print({'초기 평균 손실': round(first, 4), '후기 평균 손실': round(last, 4)})
print({'online': round(online, 4), 'predictor': round(predictor, 4), 'EMA target': round(target, 4)})
assert last < first
assert abs(target - online) > 0  # target은 online을 즉시 복사하지 않고 부드럽게 추종합니다.

## 실제 모델과의 차이

이 실습은 학습 역학만 보여 주는 toy model입니다. 실제 V-JEPA는 Transformer 표현 벡터 사이의 L1 손실을 사용하고, 타깃 인코더 모멘텀을 0.998에서 1.0까지 증가시킵니다. stop-gradient와 EMA는 예측 목표가 온라인 네트워크와 동시에 급격히 변하지 않게 합니다.